## Grid Search for Analyzer Settings

In [ ]:
# Without extension
OUTPUT_NAME = 'grid_search_node2vec_2021-07-01'

In [ ]:
# Save all generated partitions for further investigation
# (might consume a LOT of space)
save_partition = False

# Test run uses only the first review
test_run = True

weight_range = [0.125, 0.25, 0.5, 1, 2, 4, 8]

param_grid = [
    {
        'method': ['node2vec'],
        'similarity_bibliographic_coupling': weight_range.copy(),
        'similarity_cocitation': [1],
        'similarity_citation': weight_range.copy(),
        'similarity_text': weight_range.copy(),
        'walk_length': [50, 100],
        'walks_per_node': [10, 20],
        'vector_size': [16, 32],
        'topic_min_size': [1, 5, 10],
        'topics_max_number': [10, 20, 30],
        'check_cached_data': [False],      # for local testing, cache should work fine
    },
    {
        'similarity_bibliographic_coupling': weight_range.copy(),
        'similarity_cocitation': [1],
        'similarity_citation': weight_range.copy(),
        'similarity_text': weight_range.copy(),
        'method': ['louvain'],
        'resolution': [0.6, 0.8, 1],
    },
    {
        'similarity_bibliographic_coupling': [1],
        'similarity_cocitation': [0],
        'similarity_citation': weight_range.copy(),
        'similarity_text': weight_range.copy(),
        'method': ['louvain'],
        'resolution': [0.6, 0.8, 1],
    },
    {
        'similarity_bibliographic_coupling': [0],
        'similarity_cocitation': [0],
        'similarity_citation': [1],
        'similarity_text': weight_range.copy(),
        'method': ['louvain'],
        'resolution': [0.6, 0.8, 1],
    },
    {
        'method': ['lda'],
        'max_df': [0.8],
        'min_df': [0.001, 0.01, 0.1],
        'max_features': [2500, 5000, 10000],
        'topic_min_size': [5, 10],
        'topics_max_number': [10, 20],
        'check_cached_data': [False],        # for local testing, cache should work fine
    }
]

## Imports

In [ ]:
import json
import itertools
import logging
import os
import pandas as pd

from sklearn.metrics.cluster import adjusted_mutual_info_score, v_measure_score
from sklearn.model_selection import ParameterGrid

from pysrc.papers.analysis.graph import node2vec
from pysrc.papers.analyzer import PapersAnalyzer
from pysrc.papers.config import AnalyzerSettings

from utils.analysis import get_direct_references_subgraph, align_clusterings_for_sklearn
from utils.io import reload_exported_analyzer, load_analyzer, load_clustering, get_review_pmids
from utils.metrics import pd_score, reg_v_score
from utils.preprocessing import preprocess_clustering, get_clustering_level

In [ ]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

ch = logging.StreamHandler()
ch.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s %(levelname)s: %(message)s')

ch.setFormatter(formatter)

logger.addHandler(ch)

## Clustering Methods

#### Latent Dirichlet Allocation

In [ ]:
from collections import Counter
from functools import lru_cache

from pysrc.papers.analysis.text import build_corpus

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer


def cluster_lda(X, min_cluster_size, max_clusters):
    logging.debug('Looking for an appropriate number of clusters,'
                 f'min_cluster_size={min_cluster_size}, max_clusters={max_clusters}')
    r = min(max_clusters, X.shape[0]) + 1
    l = 1

    if l >= r - 2:
        return [0] * X.shape[0], None

    while l < r - 2:
        n_clusters = int((l + r) / 2)
        lda = LatentDirichletAllocation(n_components=n_clusters, random_state=0).fit(X)
        clusters = lda.transform(X).argmax(axis=1)
        clusters_counter = Counter(clusters)
        min_size = clusters_counter.most_common()[-1][1]
        num_clusters = len(clusters_counter.keys())
        if min_size < min_cluster_size or min_size > max_clusters:
            r = n_clusters + 1
        else:
            l = n_clusters

    logging.debug(f'Number of clusters = {n_clusters}')
    logging.debug('Reorder clusters by size descending')
    reorder_map = {c: i for i, (c, _) in enumerate(clusters_counter.most_common())}
    return [reorder_map[c] for c in clusters]


@lru_cache(maxsize=1000)
def preproc_lda(analyzer, subgraph, max_df, min_df, max_features):
    # Use only papers present in the subgraph
    subgraph_df = analyzer.df[analyzer.df.id.isin(subgraph.nodes)]
    node_ids = list(subgraph_df.id.values)
    
    # Build and vectorize corpus for LDA
    corpus = build_corpus(subgraph_df)
    vectorizer = CountVectorizer(max_df=max_df, min_df=min_df, max_features=max_features)
    X = vectorizer.fit_transform(corpus)
    return X, node_ids
    

def topic_analysis_lda(analyzer, subgraph, **settings):
    X, node_ids = preproc_lda(analyzer, subgraph, 
                              max_df=settings['max_df'],
                              min_df=settings['min_df'],
                              max_features=settings['max_features'])
    
    # Check the cached data
    if settings['check_cached_data']:
        n = subgraph.number_of_nodes()
        assert X.shape[0] == n
        assert X.shape[1] <= settings['max_features']
        assert list(sorted(subgraph.nodes())) == sorted(node_ids)
    
    clusters = cluster_lda(X, min_cluster_size=settings['topic_min_size'],
                           max_clusters=settings['topics_max_number'])
    
    return dict(zip(node_ids, clusters))

#### Louvain Community Clustering (restored from Git history)

In [ ]:
import community
import numpy as np
import networkx as nx

from pysrc.papers.utils import SEED

from nature_reviews.src.utils.analysis import get_similarity_func

def topic_analysis_louvain(analyzer, similarity_graph, **settings):
    """
    Performs clustering of similarity topics, merging small topics into Other component
    :param analyzer: is not used, only for consistency with other methods
    :param similarity_graph: Similarity graph
    :param settings: contains all tunable parameters
    :return: merged_partition
    """
    logging.debug('Compute aggregated similarity')
    similarity_func = get_similarity_func(settings['similarity_bibliographic_coupling'],
                                          settings['similarity_cocitation'],
                                          settings['similarity_citation'],
                                          settings['similarity_text'])    
    for _, _, d in similarity_graph.edges(data=True):
        d['similarity'] = similarity_func(d)

    logging.debug('Graph clustering via Louvain community algorithm')
    partition_louvain = community.best_partition(
        similarity_graph, weight='similarity', random_state=SEED, resolution=settings['resolution']
    )

    return partition_louvain

#### node2vec Clustering

In [ ]:
from pysrc.papers.analysis.graph import node2vec
from pysrc.papers.analysis.topics import cluster_embeddings

def topic_analysis_node2vec(analyzer, subgraph, **settings):
    """
    Rerun topic analysis based on node2vec for a given similarity graph and settings.
    """
    similarity_func = get_similarity_func(settings['similarity_bibliographic_coupling'],
                                          settings['similarity_cocitation'],
                                          settings['similarity_citation'],
                                          settings['similarity_text'])
    node_ids, node_embeddings = node2vec(subgraph,
                                         weight_func=similarity_func,
                                         walk_length=settings['walk_length'], 
                                         walks_per_node=settings['walks_per_node'], 
                                         vector_size=settings['vector_size'])
    
    if settings['check_cached_data']:
        n = subgraph.number_of_nodes()
        assert node_embeddings.shape[0] == n
        assert node_embeddings.shape[1] <= settings['vector_size']
        assert list(sorted(subgraph.nodes())) == sorted(node_ids)
    
    clusters, _ = cluster_embeddings(
        node_embeddings, settings['topic_min_size'], settings['topics_max_number']
    )
    return dict(zip(node_ids, clusters))

#### Topic Analysis Launcher (node2vec / louvain / LDA)

In [ ]:
def topic_analysis(analyzer, subgraph, method, **method_params):
    """
    Returns partition - dictionary {pmid (str): cluster (int)}
    """
    if method == 'node2vec':
        return topic_analysis_node2vec(analyzer, subgraph, **method_params)
    elif method == 'lda':
        return topic_analysis_lda(analyzer, subgraph, **method_params)
    elif method == 'louvain':
        return topic_analysis_louvain(analyzer, subgraph, **method_params)
    else:
        raise ValueError('Unknown clustering method')

## Grid Search

In [ ]:
def run_grid_search(analyzer, subgraph, ground_truth, metrics, param_grid, save_partition=False):
    # Accumulate grid results for all hierarchy levels
    grid_results = []
    partitions = []

    parameter_grid = ParameterGrid(param_grid)
    grid_size = len(parameter_grid)
    for i, param_values in enumerate(parameter_grid):
        partition = topic_analysis(analyzer, subgraph, **param_values)

        if save_partition:
            param_partition = param_values.copy()
            param_partition['partition'] = partition
            partitions.append(param_partition)

        # Iterate over hierarchy levels to avoid re-calculating same clustering for different ground truth    
        for level, ground_truth_partition in ground_truth.items():
            result = param_values.copy()
            result['level'] = level
            labels_true, labels_pred = align_clusterings_for_sklearn(partition, ground_truth_partition)
            result['n_clusters'] = len(set(labels_pred))

            # Save partition & iterate over different metrics
            for metric in metrics:
                result[metric.__name__] = metric(labels_true, labels_pred)

            grid_results.append(result)
            
        print('.', end='')
        if (i + 1) % 100 == 0:
            print(f' {i+1} / {grid_size}\n')
    print('\n')
    
    return grid_results, partitions

## Main Loop

In [ ]:
metrics = [adjusted_mutual_info_score, pd_score, reg_v_score]

In [ ]:
results_df = pd.DataFrame()
partitions_overall = [] 

review_pmids = get_review_pmids()
if test_run:
    review_pmids = [review_pmids[0]]
n_reviews = len(review_pmids)

for i, pmid in enumerate(review_pmids):
    logger.info(f'({i+1} / {n_reviews}) {pmid} - loading clustering and analyzer')
    clustering = load_clustering(pmid)
    analyzer = load_analyzer(pmid)
    logger.info(f'({i+1} / {n_reviews}) {pmid} - loaded clustering and analyzer')
    
    # Pre-calculate all hierarchy levels before grid search to avoid re-calculation of clusterings
    ground_truth = {}
    for level in range(1, get_clustering_level(clustering)):
        ground_truth[level] = preprocess_clustering(clustering, level, 
                                                    include_box_sections=False,
                                                    uniqueness_method='unique_only')
    logger.info(f'({i+1} / {n_reviews}) {pmid} - calculated ground truth for all hierarchy levels')
    
    subgraph = get_direct_references_subgraph(analyzer, pmid)
    logger.info(f'({i+1} / {n_reviews}) {pmid} - started grid search')
    
    grid_results, partitions = run_grid_search(analyzer, subgraph, ground_truth, 
                                               metrics, param_grid, save_partition=save_partition)
    
    grid_results_df = pd.DataFrame(grid_results)
    grid_results_df['pmid'] = pmid
    results_df = results_df.append(grid_results_df, ignore_index=True)
    
    partitions_overall.append({
        'pmid': pmid,
        'partitions': partitions
    })
    
    logger.info(f'({i+1} / {n_reviews}) {pmid} - done')

In [ ]:
results_df = results_df.fillna(0)
results_df = results_df.drop(columns=['check_cached_data'])

In [ ]:
results_df

In [ ]:
results_df.to_csv(f'{OUTPUT_NAME}.csv', index=False)

In [ ]:
if save_partition:
    with open(f'{OUTPUT_NAME}.json', 'w') as f:
        json.dump(partitions_overall, f)

In [ ]:
print('Grid Search - Done')

## Simple Visualization

#### Load old results if needed

In [ ]:
load_from_disk = False

In [ ]:
if load_from_disk:
    results_df = pd.read_csv(f'{OUTPUT_NAME}.csv')

#### Extract parameter columns

In [ ]:
score_columns = set([m.__name__ for m in metrics])
param_columns = list(set(results_df.columns) - score_columns - set(['level', 'n_clusters', 'pmid']))
print(param_columns)

#### Average Scores 

In [ ]:
def get_top_parameter_sets_for_method(score_df, param_cols, method, target_col, n=5):
    return score_df[score_df.method == method].groupby(param_cols)[[target_col, 'n_clusters']].mean().sort_values(by=target_col, 
                                                                                                                  ascending=False).head(n).reset_index()

In [ ]:
def get_top_mean_score_for_method(score_df, param_cols, method, target_col):
    return score_df[score_df.method == method].groupby(param_cols)[target_col].mean().sort_values(ascending=False).values[0]

In [ ]:
target_col = 'adjusted_mutual_info_score'

for method in results_df.method.unique():
    top_score = get_top_mean_score_for_method(results_df, param_columns, method, target_col)
    print(method, ':', target_col, top_score, '\n')
    print(get_top_parameter_sets_for_method(results_df, param_columns, method, target_col), '\n')

In [ ]:
mean_score_data = []
for method in results_df.method.unique():
    method_data = []
    for metric in metrics:
        top_score = get_top_mean_score_for_method(results_df, param_columns, method, metric.__name__)
        method_data.append(top_score)
    mean_score_data.append((method, *method_data))

In [ ]:
metric_names = [m.__name__ for m in metrics]
mean_score_df = pd.DataFrame(mean_score_data, columns=['method', *metric_names])
mean_score_df.head()

In [ ]:
mean_score_df.to_csv(f'{OUTPUT_NAME}_mean_scores_per_method.csv', index=False)

In [ ]:
p = mean_score_df.plot.bar(x='method', y=metric_names)
fig = p.get_figure()
fig.savefig(f'{OUTPUT_NAME}_mean_scores_per_method.png')

#### Average Scores for Different Clustering Levels

In [ ]:
def get_top_parameter_sets_for_level_and_method(score_df, param_cols, level, method, target_col, n=5):
    return score_df[(score_df.method == method) & (score_df.level == level)]\
        .groupby(param_cols)[[target_col, 'n_clusters']].mean().sort_values(by=target_col, 
                                                                            ascending=False).head(n).reset_index()

In [ ]:
def get_top_mean_score_for_level_and_method(score_df, param_cols, level, method, target_col):
    return score_df[(score_df.method == method) & (score_df.level == level)]\
        .groupby(param_cols)[target_col].mean().sort_values(ascending=False).values[0]

In [ ]:
target_col = 'adjusted_mutual_info_score'

for level in results_df.level.unique():
    print(f'LEVEL {level}')
    for method in results_df.method.unique():
        top_score = get_top_mean_score_for_level_and_method(results_df, param_columns, level, method, target_col)
        print(method, ':', target_col, top_score, '\n')
        print(get_top_parameter_sets_for_level_and_method(results_df, param_columns, level, method, target_col), '\n')

In [ ]:
level_mean_score_data = []

for level in results_df.level.unique():
    for method in results_df.method.unique():
        method_data = []
        for metric in metrics:
            top_score = get_top_mean_score_for_level_and_method(results_df, param_columns, level, method, metric.__name__)
            method_data.append(top_score)
        level_mean_score_data.append((level, method, *method_data))

In [ ]:
metric_names = [m.__name__ for m in metrics]
level_mean_score_df = pd.DataFrame(level_mean_score_data, columns=['level', 'method', *metric_names])
level_mean_score_df

In [ ]:
level_mean_score_df.to_csv(f'{OUTPUT_NAME}_mean_scores_per_method_and_level.csv', index=False)

In [ ]:
for level in level_mean_score_df.level.unique():
    p = level_mean_score_df[level_mean_score_df.level == level].plot.bar(x='method', y=metric_names, title=f'Level {level}')
    fig = p.get_figure()
    fig.savefig(f'{OUTPUT_NAME}_mean_scores_per_method_level_{level}.png')

In [ ]:
print('Visualization - Done')